# NLU Shared Task — Demo Solution 1 (Category A)

This notebook generates predictions for the AV test set using the trained Solution 1 stacking ensemble.

**Pipeline:** Raw test CSV → spaCy preprocessing → stylometric features (84) → info-theoretic features (8) → General Impostor features (5) → stacking ensemble → predictions

**Required files (download from OneDrive and place in the correct paths before running):**
- `src/solution1/models/full/model_a2.joblib`
- `src/solution1/features/gi_corpus_vectors.joblib`

In [ ]:
# Install required packages
!pip install spacy lightgbm textstat joblib tqdm scikit-learn pandas numpy scipy
!python -m spacy download en_core_web_sm

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure src is in the Python path
sys.path.insert(0, os.path.abspath('..'))

## 1. Configure Paths

Set the paths to the test data, saved models, and feature artifacts.

In [ ]:
TEST_CSV     = Path('../data/test_data/AV/test.csv')
MODEL_DIR    = Path('../src/solution1/models/full')
FEATURES_DIR = Path('../src/solution1/features')
OUTPUT_PATH  = Path('../outputs/Group_17_A.csv')
N_JOBS       = 4  # parallel workers for preprocessing

# Sanity checks
assert TEST_CSV.exists(),                                        f'Test CSV not found: {TEST_CSV}'
assert (MODEL_DIR / 'model_a2.joblib').exists(),                 f'model_a2.joblib not found in {MODEL_DIR} — download from OneDrive'
assert (FEATURES_DIR / 'gi_corpus_vectors.joblib').exists(),     f'gi_corpus_vectors.joblib not found — download from OneDrive'

print('All required files found.')
print(f'Test CSV: {pd.read_csv(TEST_CSV).shape[0]} pairs')

## 2. Generate Predictions

Runs the full pipeline end-to-end: preprocessing → feature extraction → stacking ensemble.

The GI features use the saved training corpus as the impostor pool (consistent with training).

In [ ]:
from src.solution1.predict import predict_from_csv

submission_df = predict_from_csv(
    test_csv=TEST_CSV,
    model_dir=MODEL_DIR,
    features_dir=FEATURES_DIR,
    n_jobs=N_JOBS,
)

print(f'Generated {len(submission_df)} predictions.')
print(f'Class distribution: {submission_df["prediction"].value_counts().to_dict()}')

## 3. Save Output

Saves predictions to a CSV with a single `prediction` column (0 or 1) as required by the submission spec.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved to {OUTPUT_PATH}')
display(submission_df.head(10))